# 扩大泛区图片处理演示

本Notebook演示如何对当前路径下的图片`扩大泛区.png`，去除RGB为(230, 243, 215)的区域，并统计处理结果。

## 1. 导入所需库
导入os、numpy、PIL.Image和collections.Counter等库。

In [1]:
import os
import numpy as np
from PIL import Image
from collections import Counter

## 2. 设置参数与目标颜色
定义目标RGB颜色、容差参数，并设置图片文件名为'扩大泛区.png'。

In [5]:
# 目标RGB颜色与容差设置
TARGET_RGB = (230, 243, 215)
TOL = 5  # 容差（处理压缩误差）
IMG_NAME = '扩大泛区.png'

## 3. 定位并加载图片文件
检查当前路径下是否存在'扩大泛区.png'，并使用PIL加载为RGBA格式的图片。

In [6]:
# 检查图片文件是否存在并加载
if not os.path.exists(IMG_NAME):
    raise FileNotFoundError(f"未找到图片文件: {IMG_NAME}")
im = Image.open(IMG_NAME).convert('RGBA')
arr = np.array(im)
H, W = arr.shape[:2]
print(f"图片尺寸: {W}x{H}")

图片尺寸: 939x1120


## 4. 精确去除指定颜色区域
创建掩码，精确匹配目标RGB值，将对应像素的Alpha通道设为0（透明）。

In [7]:
# 精确去除目标RGB像素
exact_mask = (arr[...,0]==TARGET_RGB[0]) & (arr[...,1]==TARGET_RGB[1]) & (arr[...,2]==TARGET_RGB[2])
arr_exact = arr.copy()
arr_exact[exact_mask, 3] = 0  # 设为透明

## 5. 容差去除指定颜色区域（处理压缩误差）
根据容差范围，匹配近似目标RGB的像素，并将其Alpha通道设为0。

In [8]:
# 容差去除目标RGB像素（处理压缩误差）
low = np.array(TARGET_RGB) - TOL
high = np.array(TARGET_RGB) + TOL
rgb = arr[...,:3].astype(np.int16)
tol_mask = (rgb>=low).all(axis=-1) & (rgb<=high).all(axis=-1)
arr_tol = arr.copy()
arr_tol[tol_mask, 3] = 0

## 6. 保存处理后的图片
将精确和容差处理后的图片分别保存为新文件。

In [9]:
# 保存处理后的图片
exact_path = '扩大泛区_removed_exact.png'
tol_path = '扩大泛区_removed_threshold.png'
Image.fromarray(arr_exact, mode='RGBA').save(exact_path)
Image.fromarray(arr_tol, mode='RGBA').save(tol_path)
print(f"精确去除结果已保存: {exact_path}")
print(f"容差去除结果已保存: {tol_path}")

精确去除结果已保存: 扩大泛区_removed_exact.png
容差去除结果已保存: 扩大泛区_removed_threshold.png


/tmp/ipykernel_15199/4229519710.py:4: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(arr_exact, mode='RGBA').save(exact_path)
/tmp/ipykernel_15199/4229519710.py:5: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(arr_tol, mode='RGBA').save(tol_path)


## 7. 统计去除像素数量
统计并输出被精确和容差去除的像素数量。

In [10]:
# 统计去除像素数量
exact_count = int(exact_mask.sum())
tol_count = int(tol_mask.sum())
print(f"精确去除像素数: {exact_count}")
print(f"容差去除像素数: {tol_count}")

精确去除像素数: 592265
容差去除像素数: 598474


## 8. 分析被去除的主要颜色
统计容差去除区域中最常见的颜色（前5名），用于分析实际被去除的颜色分布。

In [11]:
# 分析被去除的主要颜色（前5名）
removed_colors = rgb[tol_mask]
color_counts = Counter(map(tuple, removed_colors))
most_common = color_counts.most_common(5)
print("容差去除区域最常见的颜色（前5名）:")
for color, count in most_common:
    print(f"{color}: {count} 像素")

容差去除区域最常见的颜色（前5名）:
(np.int16(230), np.int16(243), np.int16(215)): 592265 像素
(np.int16(229), np.int16(243), np.int16(215)): 1818 像素
(np.int16(229), np.int16(241), np.int16(213)): 521 像素
(np.int16(229), np.int16(239), np.int16(210)): 328 像素
(np.int16(230), np.int16(242), np.int16(214)): 304 像素
